In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [6]:
base_path = Path.cwd()

geo_0 = pd.read_csv(base_path / 'geo_data_0.csv')
geo_1 = pd.read_csv(base_path / 'geo_data_1.csv')
geo_2 = pd.read_csv(base_path / 'geo_data_2.csv')

In [8]:


print(geo_0.info())
print()
print(geo_0.isna().sum())
print()
print(geo_0.duplicated().sum())
print()
print(geo_0.head())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

0

      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647


In [9]:

print(geo_1.info())
print()
print(geo_1.isna().sum())
print()
print(geo_1.duplicated().sum())
print()
print(geo_1.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

0

      id         f0         f1        f2     product
0  kBEdx -15.001348  -8.276000 -0.005876    3.179103
1  62mP7  14.272088  -3.475083  0.999183   26.953261
2  vyE1P   6.263187  -5.948386  5.001160  134.766305
3  KcrkZ -13.081196 -11.506057  4.999415  137.945408
4  AHL4O  12.702195  -8.147433  5.004363  134.766305


In [10]:
print(geo_2.info())
print()
print(geo_2.isna().sum())
print()
print(geo_2.duplicated().sum())
print()
print(geo_2.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

0

      id        f0        f1        f2     product
0  fwXo0 -1.146987  0.963328 -0.828965   27.758673
1  WJtFt  0.262778  0.269839 -2.530187   56.069697
2  ovLUW  0.194587  0.289035 -5.586433   62.871910
3  q6cA6  2.236060 -0.553760  0.930038  114.572842
4  WPMUX -0.515993  1.716266  5.899011  149.600746


The dataset contains no missing or duplicate values and is therefore considered ready for analysis.

### Preparing the Model Training Function

In [22]:
from sklearn.metrics import mean_squared_error
import numpy as np

def train_model(data):
    """
    Trains a Linear Regression model to predict oil reserve volumes.
    Computes RMSE and returns aligned predictions and validation targets.
    """
    
    # Split features and target
    features = data[['f0', 'f1', 'f2']]
    target = data['product']
    
    # Train-validation split (75:25)
    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345
    )
    
    # Train Linear Regression model
    model = LinearRegression()
    model.fit(features_train, target_train)
    
    # Predictions on validation set
    predictions = model.predict(features_valid)
    
    # Compute RMSE manually
    mse = mean_squared_error(target_valid.values, predictions)
    rmse = np.sqrt(mse)
    
    # Mean predicted volume
    mean_pred = predictions.mean()
    
    print("Average predicted volume:", mean_pred)
    print("RMSE:", rmse)
    
    # Align predictions with target index
    predictions = pd.Series(predictions, index=target_valid.index)
    
    return predictions, target_valid

### Training the Model for Each Region

In [23]:
pred_0, target_0 = train_model(geo_0)
pred_1, target_1 = train_model(geo_1)
pred_2, target_2 = train_model(geo_2)

Average predicted volume: 92.59256778438035
RMSE: 37.5794217150813
Average predicted volume: 68.728546895446
RMSE: 0.8930992867756166
Average predicted volume: 94.96504596800492
RMSE: 40.02970873393434


The results show that Regions 0 and 2 have a higher average reserve volume than Region 1, making them, in principle, more attractive for oil extraction.
However, Region 1’s RMSE is extremely low (0.89) compared to the other regions (38–40). This may be due to a peculiarity in the data or the way RMSE was calculated, and does not necessarily indicate more accurate predictions.
Therefore, when selecting the region with the highest potential profit, it is recommended to focus on the average reserve volumes and subsequent profit and risk analysis using bootstrapping, rather than interpreting RMSE in isolation.

In [12]:
print("Region 0 average:", geo_0['product'].mean())
print("Region 1 average:", geo_1['product'].mean())
print("Region 2 average:", geo_2['product'].mean())

Region 0 average: 92.50000000000001
Region 1 average: 68.82500000000002
Region 2 average: 95.00000000000004


In [13]:
BUDGET = 100_000_000       # Budget
WELLS = 200                # Number of wells to develop
REVENUE_PER_UNIT = 4500    # Revenue per thousand barrels

### Profit Calculation Function

In [14]:
def calculate_profit(predictions, target):
    """
    Calculates the potential profit for the top wells based on model predictions.

    Parameters:
    - predictions: pd.Series, predicted reserve volumes for all wells
    - target: pd.Series, actual reserve volumes

    Returns:
    - profit: float, estimated profit in dollars
    """
    
    # Sort wells by predicted volume (descending)
    sorted_preds = predictions.sort_values(ascending=False)
    
    # Select the top 200 wells
    selected = target[sorted_preds.index][:WELLS]
    
    # Total reserve volume
    total_volume = selected.sum()
    
    # Calculate revenue
    revenue = total_volume * REVENUE_PER_UNIT
    
    # Calculate profit
    profit = revenue - BUDGET
    
    return profit

### Ganancia potencial por región: 

In [24]:
profit_0 = calculate_profit(pred_0, target_0)
profit_1 = calculate_profit(pred_1, target_1)
profit_2 = calculate_profit(pred_2, target_2)

print("Region 0 profit:", profit_0)
print("Region 1 profit:", profit_1)
print("Region 2 profit:", profit_2)

Region 0 profit: 33208260.43139851
Region 1 profit: 24150866.966815114
Region 2 profit: 27103499.635998324


The analysis results show clear differences among the three evaluated regions. Region 0 has an average predicted reserve volume of approximately 92.6 thousand barrels, Region 2 has 95.0, and Region 1 has 68.7. This pattern is directly reflected in the estimated profits for the top 200 wells: $33.2 million for Region 0, $27.1 million for Region 2, and $24.2 million for Region 1.

It is observed that regions with higher average predicted volumes generate the highest profits, confirming that model-predicted volume is a strong initial indicator of economic benefit. However, these profits represent only the expected mean value; the variability of reserves may entail risks of loss if the selected wells underperform. Therefore, the final decision should not rely solely on average profits but also consider the risk of loss assessed through bootstrapping techniques.

In conclusion, Region 0 emerges as the most economically attractive option, followed by Region 2, while Region 1 is the least profitable. Nonetheless, the final selection should integrate both expected profit and probability of loss to ensure a solid and reliable investment.

### Bootstrapping


In [28]:
def bootstrap_profit(predictions, target):
    """
    Performs a bootstrap analysis to estimate the distribution of profits,
    compute the mean profit, 95% confidence interval, and risk of loss.

    Parameters:
    - predictions: pd.Series, predicted reserve volumes for all wells
    - target: pd.Series, actual reserve volumes

    Returns:
    - mean_profit: float, average profit over bootstrap samples
    - lower: float, lower bound of 95% confidence interval
    - upper: float, upper bound of 95% confidence interval
    - risk: float, probability of loss (profit < 0)
    """

    # Set random state for reproducibility
    state = np.random.RandomState(12345)

    profits = []

    for i in range(1000):
        # Sample 500 wells with replacement
        target_subsample = target.sample(n=500, replace=True, random_state=state)
        preds_subsample = predictions[target_subsample.index]

        # Calculate profit for the subsample
        profit = calculate_profit(preds_subsample, target_subsample)
        profits.append(profit)

    profits = pd.Series(profits)

    # Compute statistics
    mean_profit = profits.mean()
    lower = profits.quantile(0.025)
    upper = profits.quantile(0.975)
    risk = (profits < 0).mean()

    print("Average profit:", mean_profit)
    print("95% confidence interval:", lower, upper)
    print("Risk of loss:", risk)

    return mean_profit, lower, upper, risk

In [29]:
print("REGION 0")
bootstrap_profit(pred_0, target_0)

print("REGION 1")
bootstrap_profit(pred_1, target_1)

print("REGION 2")
bootstrap_profit(pred_2, target_2)

REGION 0
Average profit: 4259385.269105923
95% confidence interval: -1020900.9483793724 9479763.533583675
Risk of loss: 0.06
REGION 1
Average profit: 5152227.734432898
95% confidence interval: 688732.2537050088 9315475.912570495
Risk of loss: 0.01
REGION 2
Average profit: 4350083.627827557
95% confidence interval: -1288805.473297878 9697069.541802654
Risk of loss: 0.064


(np.float64(4350083.627827557),
 np.float64(-1288805.473297878),
 np.float64(9697069.541802654),
 np.float64(0.064))

The bootstrapping analysis indicates that Region 1 is the safest and most profitable option for drilling the 200 wells, with an estimated average profit of approximately $5.15 million and a risk of loss of only 1%.
Although Regions 0 and 2 could generate profits around $4.3–4.4 million, they carry higher risks (6–6.4%) and their confidence intervals include potential loss scenarios.
Therefore, Region 1 is recommended as the optimal region for well development, combining attractive returns with low investment risk.